In [1]:
import pandas as pd
import numpy as np

In [2]:
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [3]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies_df['year'] = movies_df['title'].str.extract(r'(\(\d+)\)')
movies_df['year'] = movies_df['title'].str.extract(r'(\d+)')
movies_df['title'] = movies_df['title'].str.replace(r'(\(\d+\))', '', regex = True).str.strip()
movies_df = movies_df.drop('genres', axis = 1)

In [5]:
movies_df.head()

,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


In [6]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
userInput = [
              {'title': 'Toy Story', 'rating': 8},
              {'title': 'Jumanji', 'rating': 8},
              {'title': 'Grumpier Old Men', 'rating': 5},
              {'title': 'Waiting to Exhale', 'rating': 5},
              {'title': 'Father of the Bride Part II', 'rating': 4}
            ] 
inputMovies = pd.DataFrame(userInput)
inputMovies.head()

,title,rating
0,Toy Story,8
1,Jumanji,8
2,Grumpier Old Men,5
3,Waiting to Exhale,5
4,Father of the Bride Part II,4


In [8]:
inputId = movies_df[movies_df['title'].isin(inputMovies['title'].tolist())]
inputMovies = pd.merge(inputId, inputMovies).drop('year', axis = 1)
inputMovies.head()

,movieId,title,rating
0,1,Toy Story,8
1,2,Jumanji,8
2,3,Grumpier Old Men,5
3,4,Waiting to Exhale,5
4,5,Father of the Bride Part II,4


In [9]:
userSubset = ratings_df[ratings_df['movieId'].isin(inputMovies['movieId'].tolist())].drop('timestamp' , axis = 1)

In [10]:
print()
print("userSubset_len: ", userSubset.shape[0])
print()
print()
print()
userSubset.head()


userSubset_len:  433





,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
516,5,1,4.0
560,6,2,4.0
561,6,3,5.0


In [11]:
userSubsetGroup = userSubset.groupby(['userId'])

In [12]:
userSubsetGroup.get_group((6, ))

,userId,movieId,rating
560,6,2,4.0
561,6,3,5.0
562,6,4,3.0
563,6,5,5.0


In [13]:
userSubsetGroup = sorted(userSubsetGroup, key = lambda x : len(x[1]), reverse = True)

In [14]:
userSubsetGroup[0:2:1]

[((6,),
       userId  movieId  rating
  560       6        2     4.0
  561       6        3     5.0
  562       6        4     3.0
  563       6        5     5.0),
 ((68,),
         userId  movieId  rating
  10360      68        1     2.5
  10361      68        2     2.5
  10362      68        3     2.0
  10363      68        5     2.0)]

In [15]:
from scipy.stats import pearsonr

pearsonCorrelationDict = {}

for name, group in userSubsetGroup :
    
    merged = pd.merge(inputMovies, group, on = 'movieId', suffixes = ('_input', '_other'))
    
    if len(merged) >= 2 :
        
        if merged['rating_input'].nunique() == 1 or merged['rating_other'].nunique() == 1 :
            pearsonCorrelationDict[name] = 0

        else : 
            cor,_ = pearsonr(merged['rating_input'], merged['rating_other'])

            if np.isnan(cor) :
                pearsonCorrelationDict[name] = 0
        
            else :
                pearsonCorrelationDict[name] = cor
                
    else : 
            pearsonCorrelationDict[name] = 0

In [16]:
pearson_df = pd.DataFrame.from_dict(pearsonCorrelationDict, orient = 'index')
pearson_df.columns = ['SimilarityIndex']
pearson_df['userId'] = pearson_df.index
pearson_df['userId'] = pearson_df['userId'].apply(lambda x : x[0])
pearson_df.index = range(len(pearson_df))

In [17]:
pearson_df.head()

,SimilarityIndex,userId
0,-0.301511,6
1,0.980196,68
2,-0.886621,169
3,0.230551,288
4,0.464420,414


In [18]:
topUsers = pearson_df.sort_values(by = 'SimilarityIndex', ascending = False)[0: 20]

In [19]:
topUsers.head()

,SimilarityIndex,userId
34,1.0,561
20,1.0,276
94,1.0,509
93,1.0,490
29,1.0,477


In [20]:
topUsersRating = topUsers.merge(ratings_df, on = 'userId')

In [21]:
topUsersRating['weightedRating'] = topUsersRating['SimilarityIndex'] * topUsersRating['rating']

In [22]:
topUsersRating.head()

,SimilarityIndex,userId,movieId,rating,timestamp,weightedRating
0,1.0,561,1,4.0,1491094681,4.0
1,1.0,561,2,4.0,1491094318,4.0
2,1.0,561,5,3.0,1491094488,3.0
3,1.0,561,6,4.0,1491092289,4.0
4,1.0,561,31,2.5,1491094481,2.5


In [23]:
temptopUsersRating = topUsersRating.groupby('movieId').sum()[['SimilarityIndex', 'weightedRating']]
temptopUsersRating.columns = ['sum_SimilarityIndex', 'sum_weightedRating']

In [24]:
temptopUsersRating.head()

,sum_SimilarityIndex,sum_weightedRating
movieId,,
1,16.0,68.5
2,10.0,38.0
3,7.0,20.5
5,13.0,34.5
6,8.0,31.5


In [25]:
rec = pd.DataFrame()
rec['weighted average recomendition score'] = temptopUsersRating['sum_weightedRating'] / temptopUsersRating['sum_SimilarityIndex']
rec['movieId'] = temptopUsersRating.index

In [26]:
rec = rec.sort_values(by = 'weighted average recomendition score', ascending = False)
rec.head()

,weighted average recomendition score,movieId
movieId,,
8914,5.0,8914
65642,5.0,65642
1262,5.0,1262
3677,5.0,3677
3683,5.0,3683


In [29]:
movies_df[movies_df['movieId'].isin(rec.head(20)['movieId'].tolist())].reset_index(drop = True).head()

,movieId,title,year
0,328,Tales from the Crypt Presents: Demon Knight,1995
1,1262,"Great Escape, The",1963
2,1272,Patton,1970
3,1298,Pink Floyd: The Wall,1982
4,1361,Paradise Lost: The Child Murders at Robin Hood...,1996
